<a href="https://colab.research.google.com/github/siva2513-ship-it/paddy_disease_detection/blob/main/colab_notebooks/SVM_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
from pathlib import Path

features_path = Path(
    "/content/drive/MyDrive/Paddy_Disease_Project/features"
)

X_train = np.load(features_path / "X_train.npy")
y_train = np.load(features_path / "y_train.npy")
X_valid = np.load(features_path / "X_valid.npy")
y_valid = np.load(features_path / "y_valid.npy")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_valid:", X_valid.shape)
print("y_valid:", y_valid.shape)

X_train: (8326, 512)
y_train: (8326,)
X_valid: (2081, 512)
y_valid: (2081,)


In [3]:
from fastai.vision.all import *

learn = load_learner(
    "/content/drive/MyDrive/Paddy_Disease_Project/resnet34_paddy_baseline.pkl"
)

print(learn.model)
print(learn.dls.vocab)

/usr/local/lib/python3.13/dist-packages/fastai/learner.py:456: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")


Sequential(
  (0): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  

In [4]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import time

classes = [
    'bacterial_leaf_blight',
    'bacterial_leaf_streak',
    'bacterial_panicle_blight',
    'blast',
    'brown_spot',
    'dead_heart',
    'downy_mildew',
    'hispa',
    'normal',
    'tungro'
]

svm_model = SVC(
    kernel='rbf',
    C=10,
    gamma='scale',
    probability=True,
    random_state=42
)

start_time = time.time()
svm_model.fit(X_train, y_train)
svm_train_time = time.time() - start_time

start_time = time.time()
svm_preds = svm_model.predict(X_valid)
svm_inference_time = time.time() - start_time

svm_accuracy = accuracy_score(y_valid, svm_preds)

print(f"SVM Training time: {svm_train_time:.4f} seconds")
print(f"SVM Inference time: {svm_inference_time:.4f} seconds")
print(f"\nSVM Accuracy: {svm_accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(
    y_valid,
    svm_preds,
    target_names=classes,
    digits=4
))

SVM Training time: 102.6690 seconds
SVM Inference time: 10.2296 seconds

SVM Accuracy: 76.93%

Classification Report:
                          precision    recall  f1-score   support

   bacterial_leaf_blight     0.6901    0.5568    0.6164        88
   bacterial_leaf_streak     0.8148    0.5641    0.6667        78
bacterial_panicle_blight     0.8980    0.6111    0.7273        72
                   blast     0.7653    0.8427    0.8021       356
              brown_spot     0.7074    0.7000    0.7037       190
              dead_heart     0.8598    0.8981    0.8785       314
            downy_mildew     0.6628    0.5044    0.5729       113
                   hispa     0.7560    0.7768    0.7662       327
                  normal     0.7738    0.8427    0.8068       337
                  tungro     0.7333    0.7476    0.7404       206

                accuracy                         0.7693      2081
               macro avg     0.7661    0.7044    0.7281      2081
            weighted a

In [5]:
import joblib

model_path = "/content/drive/MyDrive/Paddy_Disease_Project/models"
!mkdir -p "$model_path"

joblib.dump(
    svm_model,
    f"{model_path}/model_2_svm.pkl"
)

print("SVM model saved successfully.")

SVM model saved successfully.
